# Chapter 4 — Module Patterns (Practice)

Work through these exercises **after reading** `notes/ch04-module-patterns.md`.

Each exercise states the *decision you're practicing*, gives a stub cell to fill in, and is followed by a pre-written **verification cell** — run it to grade yourself. Hand-write your answers in your working copy under `solutions/` (this `template/` copy stays pristine), and don't peek at `solved/` until the verification passes or you're genuinely stuck.

In [1]:
# ============================================================
# TOPIC: nn.Module patterns — registration, containers, init, freezing
# MATH:  Linear: y = x W^T + b;  init: Var(w) = 1/d_in keeps Var(y) = Var(x)
# REF:   B00 ch04 notes — module-patterns
# ============================================================

# --- Imports ---
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- Reproducibility & device ---
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch version : {torch.__version__}")
print(f"device        : {device}  (every exercise here runs fine on CPU)")

torch version : 2.13.0
device        : cpu  (every exercise here runs fine on CPU)


## Exercise 1 — Manual `Linear`

Rebuild `nn.Linear` from scratch: a `weight` Parameter of shape **`(out_features, in_features)`** (each row = one output neuron's recipe), a `bias` Parameter of zeros, and the forward `x @ W.T + b`.

The verification first checks the notes §3 dry-run numbers by hand, then diffs your layer against a real `nn.Linear` carrying identical weights.

**Decision you're practicing:** `nn.Parameter` as "tensor + payroll registration" — and why the weight is `(out, in)`, forcing the transpose in forward.

In [2]:
class ManualLinear(nn.Module):
    """A from-scratch nn.Linear: y = x @ W.T + b.

    weight: (out_features, in_features) Parameter, init N(0, 1/sqrt(in_features))
    bias:   (out_features,) Parameter, init zeros
    """
    def __init__(self, in_features, out_features):
        super().__init__()
        # nn.Parameter = tensor + requires_grad + REGISTRATION on assignment
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * in_features ** -0.5)
        self.bias = nn.Parameter(torch.zeros(out_features))

    def forward(self, inputs):
        """inputs: (..., in_features) -> (..., out_features)"""
        return inputs @ self.weight.T + self.bias      # rows are recipes → transpose needed

demo = ManualLinear(3, 2)
print(f"registered parameters: {[name for name, _ in demo.named_parameters()]}")
print(f"weight shape: {tuple(demo.weight.shape)}  # (out_features, in_features)")

registered parameters: ['weight', 'bias']
weight shape: (2, 3)  # (out_features, in_features)


**Verification**

In [3]:
# --- Verification: Exercise 1 ---
manual = ManualLinear(3, 2)
param_names = [name for name, _ in manual.named_parameters()]
assert sorted(param_names) == ["bias", "weight"], f"expected weight+bias registered, got {param_names}"
assert manual.weight.shape == (2, 3), f"weight must be (out, in) = (2, 3), got {tuple(manual.weight.shape)}"

# the notes §3 dry-run, verbatim
with torch.no_grad():
    manual.weight.copy_(torch.tensor([[1.0, 0.0, -1.0], [0.5, 0.5, 0.5]]))
    manual.bias.copy_(torch.tensor([0.1, -0.1]))
dry_run_output = manual(torch.tensor([[1.0, 2.0, 3.0]]))
assert torch.allclose(dry_run_output, torch.tensor([[-1.9, 2.9]]), atol=1e-6), \
    f"dry-run mismatch: got {dry_run_output.tolist()}, want [[-1.9, 2.9]]"
print(f"notes dry-run reproduced: {dry_run_output.tolist()} ✓")

# and against the real thing, sharing weights
reference = nn.Linear(3, 2)
with torch.no_grad():
    reference.weight.copy_(manual.weight)
    reference.bias.copy_(manual.bias)
test_inputs = torch.randn(5, 3)
assert torch.allclose(manual(test_inputs), reference(test_inputs), atol=1e-6), "outputs differ from nn.Linear"
print("matches nn.Linear on random inputs ✓")
print("Exercise 1 passed ✓")

notes dry-run reproduced: [[-1.899999976158142, 2.9000000953674316]] ✓
matches nn.Linear on random inputs ✓
Exercise 1 passed ✓


## Exercise 2 — Registration Detective

`ToyBlock` holds the three kinds of state from notes §2: a Parameter, a buffer, and a plain tensor attribute. Predict, for each, the triple `(in_state_dict, in_parameters, converted_by_to)` — then the verification runs `model.to(torch.float64)` and checks reality.

**Decision you're practicing:** Parameter vs `register_buffer` vs plain attribute — what gets saved, trained, and moved.

In [4]:
class ToyBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(2))                      # learned
        self.register_buffer("positions", torch.linspace(0, 1, 4))   # fixed model state
        self.stash = torch.zeros(3)                                   # plain attribute!

block = ToyBlock()

predictions = {
    #             (in_state_dict, in_parameters, converted_by_to)
    "scale":     (True,  True,  True),    # Parameter: payroll + inventory + moved
    "positions": (True,  False, True),    # buffer: inventory only, but still moved
    "stash":     (False, False, False),   # plain tensor: invisible to everything
}
print(f"state_dict keys : {list(block.state_dict().keys())}")
print(f"parameters      : {[name for name, _ in block.named_parameters()]}")

state_dict keys : ['scale', 'positions']
parameters      : ['scale']


**Verification**

In [5]:
# --- Verification: Exercise 2 ---
state_keys = set(block.state_dict().keys())
param_names = {name for name, _ in block.named_parameters()}
block.to(torch.float64)

truth = {
    "scale":     ("scale" in state_keys,     "scale" in param_names,     block.scale.dtype == torch.float64),
    "positions": ("positions" in state_keys, "positions" in param_names, block.positions.dtype == torch.float64),
    "stash":     ("stash" in state_keys,     "stash" in param_names,     block.stash.dtype == torch.float64),
}
explanations = {
    "scale":     "Parameter → saved, trained, moved",
    "positions": "buffer → saved and moved, but never trained",
    "stash":     "plain attribute → INVISIBLE: not saved, not trained, left behind by .to()",
}
wrong = 0
for name, actual in truth.items():
    assert predictions[name] is not None, f"no prediction for {name!r}"
    mark = "✓" if tuple(predictions[name]) == actual else "✗"
    wrong += tuple(predictions[name]) != actual
    print(f"{mark} {name:10s} truth={str(actual):21s} — {explanations[name]}")
assert wrong == 0, f"{wrong} prediction(s) wrong — revisit the payroll/inventory table in notes §2"
print(f"\nthe silent bug in one line: after .to(float64), stash is still {block.stash.dtype}")
print("Exercise 2 passed ✓")

✓ scale      truth=(True, True, True)    — Parameter → saved, trained, moved
✓ positions  truth=(True, False, True)   — buffer → saved and moved, but never trained
✓ stash      truth=(False, False, False) — plain attribute → INVISIBLE: not saved, not trained, left behind by .to()

the silent bug in one line: after .to(float64), stash is still torch.float32
Exercise 2 passed ✓


## Exercise 3 — The Invisible-Parameters Bug

`BrokenStack` stores its layers in a **plain Python list**. It runs. It produces outputs. It will never learn anything. Predict `broken_param_count`, then write `FixedStack` (one-word fix) and watch the difference.

**Decision you're practicing:** registration only happens for Modules/Parameters assigned to attributes — a *list of* modules is just a list.

In [6]:
class BrokenStack(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = [nn.Linear(8, 8) for _ in range(3)]     # ← plain list: nothing registers

    def forward(self, x):
        for layer in self.layers:
            x = torch.relu(layer(x))
        return x

broken_param_count_prediction = 0     # the payroll is empty — PyTorch never saw the layers

class FixedStack(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(8, 8) for _ in range(3)])   # ← the one word

    def forward(self, x):
        for layer in self.layers:
            x = torch.relu(layer(x))
        return x

broken, fixed = BrokenStack(), FixedStack()
print(f"broken parameters: {sum(p.numel() for p in broken.parameters())}")
print(f"fixed  parameters: {sum(p.numel() for p in fixed.parameters())}   # 3 × (8·8 + 8) = 216")
print(f"broken state_dict: {len(broken.state_dict())} entries — a checkpoint of NOTHING")
print(f"fixed  state_dict: {len(fixed.state_dict())} entries")

broken parameters: 0
fixed  parameters: 216   # 3 × (8·8 + 8) = 216
broken state_dict: 0 entries — a checkpoint of NOTHING
fixed  state_dict: 6 entries


**Verification**

In [7]:
# --- Verification: Exercise 3 ---
broken, fixed = BrokenStack(), FixedStack()
broken_count = sum(p.numel() for p in broken.parameters())
fixed_count = sum(p.numel() for p in fixed.parameters())

assert broken_param_count_prediction == broken_count == 0, \
    f"broken stack reports {broken_count} params (you predicted {broken_param_count_prediction})"
assert fixed_count == 3 * (8 * 8 + 8), f"fixed stack should have 216 params, got {fixed_count}"
assert len(broken.state_dict()) == 0 and len(fixed.state_dict()) == 6

# the punchline: an optimizer over the broken model can't even be built
try:
    torch.optim.SGD(broken.parameters(), lr=0.1)
    raise AssertionError("optimizer accepted an empty parameter list?!")
except ValueError as err:
    print(f"optimizer over BrokenStack → ValueError: {err}")

# the forward still runs — that is exactly what makes this bug expensive
output = broken(torch.randn(2, 8))
print(f"...yet broken(x) happily returns shape {tuple(output.shape)} — it 'works', it just never learns")
print("Exercise 3 passed ✓")

optimizer over BrokenStack → ValueError: optimizer got an empty parameter list
...yet broken(x) happily returns shape (2, 8) — it 'works', it just never learns
Exercise 3 passed ✓


## Exercise 4 — The Same MLP, Three Ways

Build the identical 8 → 16 → 16 → 4 ReLU MLP with each container:

1. `MLPSequential` — one `nn.Sequential`, no hand-written flow
2. `MLPModuleList` — an `nn.ModuleList` of the three Linears, ReLU applied in your loop (not after the last layer!)
3. `MLPModuleDict` — an `nn.ModuleDict` with keys `"input"`, `"hidden"`, `"output"`

The verification copies one set of weights into all three and demands bit-identical outputs.

**Decision you're practicing:** the container decision tree — straight pipe → Sequential; hand-written flow → ModuleList; named branches → ModuleDict.

In [8]:
class MLPSequential(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(8, 16), nn.ReLU(),
            nn.Linear(16, 16), nn.ReLU(),
            nn.Linear(16, 4),
        )

    def forward(self, x):
        return self.net(x)                       # the pipe IS the forward

class MLPModuleList(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(8, 16), nn.Linear(16, 16), nn.Linear(16, 4)])

    def forward(self, x):
        for index, layer in enumerate(self.layers):
            x = layer(x)
            if index < len(self.layers) - 1:     # ReLU between, not after the last
                x = torch.relu(x)
        return x

class MLPModuleDict(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks = nn.ModuleDict({
            "input":  nn.Linear(8, 16),
            "hidden": nn.Linear(16, 16),
            "output": nn.Linear(16, 4),
        })

    def forward(self, x):
        x = torch.relu(self.blocks["input"](x))
        x = torch.relu(self.blocks["hidden"](x))
        return self.blocks["output"](x)

for cls in (MLPSequential, MLPModuleList, MLPModuleDict):
    model = cls()
    print(f"{cls.__name__:14s}: {sum(p.numel() for p in model.parameters())} params")

MLPSequential : 484 params
MLPModuleList : 484 params
MLPModuleDict : 484 params


**Verification**

In [9]:
# --- Verification: Exercise 4 ---
seq_model, list_model, dict_model = MLPSequential(), MLPModuleList(), MLPModuleDict()

expected_params = (8 * 16 + 16) + (16 * 16 + 16) + (16 * 4 + 4)      # 484
for model in (seq_model, list_model, dict_model):
    count = sum(p.numel() for p in model.parameters())
    assert count == expected_params, f"{type(model).__name__}: {count} params, expected {expected_params}"

# copy the Sequential's weights into the other two, then demand identical outputs
seq_linears = [m for m in seq_model.net if isinstance(m, nn.Linear)]
with torch.no_grad():
    for source, target in zip(seq_linears, list_model.layers):
        target.load_state_dict(source.state_dict())
    for source, key in zip(seq_linears, ["input", "hidden", "output"]):
        dict_model.blocks[key].load_state_dict(source.state_dict())

test_inputs = torch.randn(6, 8)
reference = seq_model(test_inputs)
assert torch.allclose(list_model(test_inputs), reference, atol=1e-6), \
    "ModuleList output differs — ReLU after the last layer, maybe?"
assert torch.allclose(dict_model(test_inputs), reference, atol=1e-6), "ModuleDict output differs"
print(f"all three containers agree on output shape {tuple(reference.shape)}, {expected_params} params each ✓")
print("Exercise 4 passed ✓")

all three containers agree on output shape (6, 4), 484 params each ✓
Exercise 4 passed ✓


## Exercise 5 — Init Surgery with `model.apply`

Build a 5-layer, width-20 linear stack and initialize every `nn.Linear` two ways using **`model.apply` + `isinstance` + `torch.no_grad`**: once with $\sigma_w = 1.0$ and once with $\sigma_w = 1/\sqrt{20}$. Then trace the activation std layer by layer and watch one stack explode (each layer multiplies std by $\sqrt{20} \approx 4.47$) while the other stays put.

**Decision you're practicing:** the `apply`-based init idiom, why it must sit inside `no_grad` (chapter 3's leaf rule), and the $1/\sqrt{d_{\text{in}}}$ variance law.

In [10]:
def make_stack(num_layers=5, width=20):
    """A plain stack of bias-free Linear(width, width) layers."""
    return nn.Sequential(*[nn.Linear(width, width, bias=False) for _ in range(num_layers)])

def init_all_linears(model, std):
    """Re-initialize every nn.Linear's weight to N(0, std) via model.apply."""
    def _init(module):
        if isinstance(module, nn.Linear):
            with torch.no_grad():                       # in-place write on a requiring leaf → must hide from autograd
                module.weight.normal_(mean=0.0, std=std)
    model.apply(_init)                                  # walks EVERY submodule, children first

def layer_stds(model, inputs):
    """Feed inputs through layer by layer; return the list of activation stds."""
    stds = []
    activation = inputs
    for layer in model:
        activation = layer(activation)                  # shape: (batch, width) throughout
        stds.append(activation.std().item())
    return stds

torch.manual_seed(0)
probe = torch.randn(512, 20)                            # big batch → stable statistics

exploding_stack = make_stack()
init_all_linears(exploding_stack, std=1.0)
exploding = layer_stds(exploding_stack, probe)

stable_stack = make_stack()
init_all_linears(stable_stack, std=20 ** -0.5)
stable = layer_stds(stable_stack, probe)

print(f"{'layer':>6s} {'std=1.0':>12s} {'std=1/sqrt(20)':>16s}")
for index, (big, small) in enumerate(zip(exploding, stable)):
    print(f"{index:>6d} {big:>12.2f} {small:>16.4f}")
print("\n→ each std=1.0 layer multiplies the signal by ≈ sqrt(20) ≈ 4.47; five layers ≈ 1800×")

 layer      std=1.0   std=1/sqrt(20)
     0         4.66           1.0527
     1        22.54           1.0767
     2        98.10           1.0681
     3       501.26           1.1726
     4      2216.01           1.1319

→ each std=1.0 layer multiplies the signal by ≈ sqrt(20) ≈ 4.47; five layers ≈ 1800×


**Verification**

In [11]:
# --- Verification: Exercise 5 ---
torch.manual_seed(0)
probe = torch.randn(512, 20)

check_stack = make_stack()
init_all_linears(check_stack, std=1.0)
exploding = layer_stds(check_stack, probe)
assert exploding is not None and len(exploding) == 5, "layer_stds must return 5 values"
assert exploding[-1] > 100, f"std=1.0 should explode past 100 by layer 5, got {exploding[-1]:.1f}"
for index in range(1, 5):
    assert exploding[index] > exploding[index - 1], "each layer should amplify the signal"

init_all_linears(check_stack, std=20 ** -0.5)
stable = layer_stds(check_stack, probe)
assert 0.2 < stable[-1] < 5.0, f"1/sqrt(d_in) init should keep std near 1, got {stable[-1]:.3f}"

# apply() must have reached every layer — no stragglers with the old init
weight_stds = [layer.weight.std().item() for layer in check_stack]
assert all(abs(w - 20 ** -0.5) < 0.05 for w in weight_stds), \
    f"some layer was missed by apply(): weight stds = {weight_stds}"
print(f"exploded to {exploding[-1]:.0f} vs stable at {stable[-1]:.3f} ✓")
print("Exercise 5 passed ✓")

exploded to 2216 vs stable at 0.975 ✓
Exercise 5 passed ✓


## Exercise 6 — Residual Connections Need `ModuleList`

Build `DeepNet(layer_sizes, use_shortcut)`: an `nn.ModuleList` of `nn.Sequential(nn.Linear(...), nn.GELU())` blocks, where the forward adds a **residual shortcut** (`x = x + block(x)`) whenever input and output shapes match. This flow is exactly what `nn.Sequential` alone cannot express — and the shortcut's effect on gradient flow is dramatic.

Then implement `first_layer_grad_mean` to measure the mean absolute gradient reaching the *first* layer.

**Decision you're practicing:** ModuleList + hand-written forward for non-pipe data flow — and *why* residuals exist (the `+ x` gives gradients a highway past the blocks: $\frac{\partial}{\partial x}(x + f(x)) = 1 + f'(x)$).

In [12]:
class DeepNet(nn.Module):
    """A deep stack of Linear+GELU blocks with optional residual shortcuts."""
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(size_in, size_out), nn.GELU())
            for size_in, size_out in zip(layer_sizes[:-1], layer_sizes[1:])
        ])

    def forward(self, x):
        for block in self.layers:
            block_output = block(x)
            if self.use_shortcut and x.shape == block_output.shape:
                x = x + block_output              # the gradient highway: d/dx (x + f(x)) = 1 + f'(x)
            else:
                x = block_output                  # shapes differ (e.g. final 3→1): no shortcut possible
        return x

def first_layer_grad_mean(model, inputs, target):
    """Run forward + MSE loss + backward; return mean |grad| of the FIRST Linear's weight."""
    output = model(inputs)
    loss = F.mse_loss(output, target)
    loss.backward()
    first_linear_weight = model.layers[0][0].weight     # path: ModuleList[0] → Sequential[0] → Linear
    return first_linear_weight.grad.abs().mean().item()

LAYER_SIZES = [3, 3, 3, 3, 3, 1]
sample_input = torch.tensor([[1.0, 0.0, -1.0]])
sample_target = torch.tensor([[0.0]])

torch.manual_seed(42)
plain_net = DeepNet(LAYER_SIZES, use_shortcut=False)
plain_grad = first_layer_grad_mean(plain_net, sample_input, sample_target)

torch.manual_seed(42)                                    # identical init for a fair fight
shortcut_net = DeepNet(LAYER_SIZES, use_shortcut=True)
shortcut_grad = first_layer_grad_mean(shortcut_net, sample_input, sample_target)

print(f"first-layer mean |grad| WITHOUT shortcuts: {plain_grad:.8f}")
print(f"first-layer mean |grad| WITH    shortcuts: {shortcut_grad:.8f}")
print(f"the shortcut multiplied the training signal by ≈ {shortcut_grad / plain_grad:.0f}×")

first-layer mean |grad| WITHOUT shortcuts: 0.00012919
first-layer mean |grad| WITH    shortcuts: 0.00239069
the shortcut multiplied the training signal by ≈ 19×


**Verification**

In [13]:
# --- Verification: Exercise 6 ---
LAYER_SIZES = [3, 3, 3, 3, 3, 1]
sample_input = torch.tensor([[1.0, 0.0, -1.0]])
sample_target = torch.tensor([[0.0]])

torch.manual_seed(42)
plain_net = DeepNet(LAYER_SIZES, use_shortcut=False)
assert sum(p.numel() for p in plain_net.parameters()) == 4 * (3 * 3 + 3) + (3 * 1 + 1), \
    "parameter count is off — did the layers register? (ModuleList, not a plain list)"
assert plain_net(sample_input).shape == (1, 1), "output should be (1, 1)"

plain_grad = first_layer_grad_mean(plain_net, sample_input, sample_target)
torch.manual_seed(42)
shortcut_net = DeepNet(LAYER_SIZES, use_shortcut=True)
shortcut_grad = first_layer_grad_mean(shortcut_net, sample_input, sample_target)

assert plain_grad is not None and shortcut_grad is not None, "implement first_layer_grad_mean"
assert shortcut_grad > 5 * plain_grad, \
    f"shortcuts should massively boost the first layer's gradient ({shortcut_grad:.2e} vs {plain_grad:.2e})"
print(f"gradient boost from residuals: {shortcut_grad / plain_grad:.0f}× ✓")
print("Exercise 6 passed ✓")

gradient boost from residuals: 19× ✓
Exercise 6 passed ✓


## Exercise 7 — Count, Freeze, and the `eval()` Misconception

`TinyClassifier` (given) is embedding → encoder → head. Implement:

1. `count_parameters(model, trainable_only=False)`
2. `freeze_all_but(model, keep_prefix)` — set `requires_grad` to `True` only for parameters whose name starts with `keep_prefix`, `False` for all others

The verification freezes everything but the head, trains one step, and proves the body never moved. Then it puts a *fully trainable* copy into `model.eval()` mode, trains it anyway, and proves **eval() froze nothing**.

**Decision you're practicing:** freezing = `requires_grad_(False)` + optimizer over requiring params only; `eval()` changes layer behavior, never trainability.

In [14]:
class TinyClassifier(nn.Module):
    def __init__(self, vocab_size=20, embed_dim=8, num_classes=3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.encoder = nn.Linear(embed_dim, embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, token_ids):
        pooled = self.embedding(token_ids).mean(dim=1)     # (batch, embed_dim)
        return self.head(torch.relu(self.encoder(pooled)))

def count_parameters(model, trainable_only=False):
    """Total number of parameter elements; only requiring ones if trainable_only."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable_only)

def freeze_all_but(model, keep_prefix):
    """requires_grad=True only for parameter names starting with keep_prefix; False otherwise."""
    for name, param in model.named_parameters():
        param.requires_grad_(name.startswith(keep_prefix))
        print(f"  {'train ' if param.requires_grad else 'FROZEN'}  {name}")

model = TinyClassifier()
print(f"total params     : {count_parameters(model)}   # 20·8 + (8·8+8) + (8·3+3) = 259")
freeze_all_but(model, "head")
print(f"trainable params : {count_parameters(model, trainable_only=True)}   # just the head: 8·3+3 = 27")

total params     : 259   # 20·8 + (8·8+8) + (8·3+3) = 259
  FROZEN  embedding.weight
  FROZEN  encoder.weight
  FROZEN  encoder.bias
  train   head.weight
  train   head.bias
trainable params : 27   # just the head: 8·3+3 = 27


**Verification**

In [15]:
# --- Verification: Exercise 7 ---
model = TinyClassifier()
assert count_parameters(model) == 259, f"total should be 259, got {count_parameters(model)}"

freeze_all_but(model, "head")
assert count_parameters(model, trainable_only=True) == 27, "only the head (27 params) should remain trainable"

# one real training step: the frozen body must not move
token_batch = torch.randint(0, 20, (6, 4))
labels = torch.randint(0, 3, (6,))
embedding_before = model.embedding.weight.clone()
encoder_before = model.encoder.weight.clone()
head_before = model.head.weight.clone()

optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=0.1)
loss = F.cross_entropy(model(token_batch), labels)
loss.backward()
optimizer.step()

assert torch.equal(model.embedding.weight, embedding_before), "frozen embedding moved!"
assert torch.equal(model.encoder.weight, encoder_before), "frozen encoder moved!"
assert not torch.equal(model.head.weight, head_before), "the head should have trained"
print("frozen body untouched, head trained ✓")

# the misconception: eval() does NOT freeze
eval_model = TinyClassifier()
eval_model.eval()                                        # behavior switch, not a freeze
eval_head_before = eval_model.head.weight.clone()
eval_optimizer = torch.optim.AdamW(eval_model.parameters(), lr=0.1)
F.cross_entropy(eval_model(token_batch), labels).backward()
eval_optimizer.step()
assert not torch.equal(eval_model.head.weight, eval_head_before), \
    "weights should STILL change in eval mode — eval() is not freezing"
print("eval() model trained anyway — eval() switches behavior, requires_grad_(False) freezes ✓")
print("Exercise 7 passed ✓")

  FROZEN  embedding.weight
  FROZEN  encoder.weight
  FROZEN  encoder.bias
  train   head.weight
  train   head.bias


frozen body untouched, head trained ✓
eval() model trained anyway — eval() switches behavior, requires_grad_(False) freezes ✓
Exercise 7 passed ✓


---
## Done!

Compare your work against `solved/ch04-module-patterns-solved.ipynb`.

You can now *assemble* models with the right containers and provably control what trains. Next: **ch05 — NLP Layer & Loss Toolbox** — the specific layers those containers hold (`Embedding`, `LayerNorm`, `Dropout`, the activation zoo) and the loss functions that train them.